# Scenario 4 — Explanation-Augmented (CoT) Fine-Tuning
## Notebook 1 of 2 — Encoder Baseline (BERT-base, Label-Only FT)

**Role in Scenario 4:** Reference ceiling — shows what a specialist classifier achieves.
BERT produces no explanations. It exists here only to frame how good Notebook 2 gets.

### Shared Conditions (identical in both notebooks)
| Factor | Value |
|---|---|
| Dataset | `puyang2025/seven-phishing-email-datasets` — SpamAssassin subset |
| Split | 80 / 10 / 10 stratified, `random_state=42` |
| Metrics | Accuracy, Precision, Recall, F1, ms/sample |
| Max token length | 256 |
| Epochs | 3 |
| Early stopping | Val F1, patience=2 |

### Fixes vs original
- Proper 80/10/10 split (original used first-200 train rows as validation)
- Early stopping on val F1 (original had none)
- Same evaluation function and metric format as Notebook 2


In [ ]:
!nvidia-smi
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall -q
!pip install transformers accelerate datasets scikit-learn pandas numpy tqdm -q


In [1]:
import os, re, time, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                              precision_score, recall_score, classification_report)
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm

# ── Shared constants (same in both notebooks) ──
SEED     = 42
MAX_LEN  = 256
EPOCHS   = 3
PATIENCE = 2
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── Shared evaluation function (same in both notebooks) ──
def evaluate(y_true, y_pred, name="", ms=None):
    result = {
        "Model"     : name,
        "Accuracy"  : f"{accuracy_score(y_true, y_pred):.4f}",
        "Precision" : f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall"    : f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
        "F1"        : f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}",
    }
    if ms:
        result["ms/sample"] = f"{ms:.2f}"
    return result

# ── Dataset class ──
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.enc = tokenizer(
            list(texts), padding="max_length", truncation=True,
            max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.labels[i]


Device : cuda
GPU    : Tesla T4


In [3]:
# ── Shared dataset loading (same in both notebooks) ──
print("Loading puyang2025/seven-phishing-email-datasets...")
ds     = load_dataset("puyang2025/seven-phishing-email-datasets", split="train")
df_all = ds.to_pandas()

# Inspect actual columns
print("Columns:", df_all.columns.tolist())

# Build text from whatever columns exist
if "body" in df_all.columns:
    df_all["text"] = (df_all["subject"].fillna("") + " " + df_all["body"].fillna("")).str.strip()
elif "text" in df_all.columns:
    df_all["text"] = df_all["text"].fillna("").str.strip()
else:
    # fallback: concatenate all string columns except label/dataset_name
    str_cols = [c for c in df_all.columns if c not in ("label", "dataset_name")
                and df_all[c].dtype == object]
    print("Using string cols for text:", str_cols)
    df_all["text"] = df_all[str_cols].fillna("").agg(" ".join, axis=1).str.strip()

df_all = (df_all[["text", "label", "dataset_name"]]
          .drop_duplicates("text").dropna().reset_index(drop=True))
df_all["label"] = df_all["label"].astype(int)

print(f"Total rows : {len(df_all):,}")
print(df_all["dataset_name"].value_counts().to_string())

Loading puyang2025/seven-phishing-email-datasets...
Columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']
Total rows : 161,480
dataset_name
TREC-05     44259
TREC-07     42746
CEAS-08     30728
Enron       23799
TREC-06     13081
Assassin     4576
Ling         2291


In [4]:
# ── Shared split (same in both notebooks) ──
# 80 / 10 / 10  stratified  random_state=42
df_assassin = df_all[df_all["dataset_name"] == "Assassin"].reset_index(drop=True)

df_tr,  df_tmp = train_test_split(df_assassin, test_size=0.20,
                                   stratify=df_assassin["label"], random_state=SEED)
df_val, df_te  = train_test_split(df_tmp,       test_size=0.50,
                                   stratify=df_tmp["label"],       random_state=SEED)

X_tr,  y_tr  = df_tr["text"].values,  df_tr["label"].values
X_val, y_val = df_val["text"].values, df_val["label"].values
X_te,  y_te  = df_te["text"].values,  df_te["label"].values

print(f"Train : {len(df_tr):,}  |  positive rate: {y_tr.mean():.3f}")
print(f"Val   : {len(df_val):,}  |  positive rate: {y_val.mean():.3f}")
print(f"Test  : {len(df_te):,}  |  positive rate: {y_te.mean():.3f}")


Train : 3,660  |  positive rate: 0.293
Val   : 458  |  positive rate: 0.295
Test  : 458  |  positive rate: 0.293


In [7]:
TRAINED_MODELS = {}
ALL_RESULTS    = []

MODELS = {
    "bert-base-uncased"      : {"batch": 16, "lr": 2e-5},
    "bert-base-cased"        : {"batch": 16, "lr": 2e-5},
    "roberta-base"           : {"batch": 16, "lr": 2e-5},
    "distilbert-base-uncased": {"batch": 16, "lr": 2e-5},
}

for MODEL_ID, cfg in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Training: {MODEL_ID}")
    print(f"{'='*60}")

    tok   = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(
                MODEL_ID, num_labels=2).to(DEVICE)

    tr_dl  = DataLoader(EmailDataset(X_tr,  y_tr,  tok), batch_size=cfg["batch"], shuffle=True)
    val_dl = DataLoader(EmailDataset(X_val, y_val, tok), batch_size=cfg["batch"]*2)

    opt   = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, len(tr_dl)//5, len(tr_dl)*EPOCHS)

    best_val_f1, patience_ctr, best_state = 0.0, 0, None

    for ep in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0
        for be, lbl in tqdm(tr_dl, desc=f"{MODEL_ID} ep{ep}"):
            be  = {k: v.to(DEVICE) for k, v in be.items()}
            out = model(**be, labels=lbl.to(DEVICE))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            train_loss += out.loss.item()

        model.eval()
        val_preds = []
        with torch.no_grad():
            for be, _ in val_dl:
                val_preds.extend(
                    model(**{k: v.to(DEVICE) for k, v in be.items()})
                    .logits.argmax(-1).cpu().numpy())

        val_f1 = f1_score(y_val, val_preds, average="binary", zero_division=0)
        print(f"  Epoch {ep} | train_loss={train_loss/len(tr_dl):.4f} | val_F1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1  = val_f1
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            print(f"    ✓ New best val F1: {best_val_f1:.4f}")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stop at epoch {ep}")
                break

    print(f"\nBest val F1: {best_val_f1:.4f}")
    TRAINED_MODELS[MODEL_ID] = {"best_state": best_state, "tok": tok}
    del model; torch.cuda.empty_cache()


Training: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bert-base-uncased ep1:   0%|          | 0/229 [00:00<?, ?it/s]

  Epoch 1 | train_loss=0.1843 | val_F1=0.9701
    ✓ New best val F1: 0.9701


...

  Epoch 1 | train_loss=0.2047 | val_F1=0.9668
    ✓ New best val F1: 0.9668


distilbert-base-uncased ep2:   0%|          | 0/229 [00:00<?, ?it/s]

  Epoch 2 | train_loss=0.0374 | val_F1=0.9701
    ✓ New best val F1: 0.9701


distilbert-base-uncased ep3:   0%|          | 0/229 [00:00<?, ?it/s]

  Epoch 3 | train_loss=0.0145 | val_F1=0.9701

Best val F1: 0.9701


In [9]:
# Load best checkpoints → evaluate all models on test set
for MODEL_ID, saved in TRAINED_MODELS.items():
    tok   = saved["tok"]
    model = AutoModelForSequenceClassification.from_pretrained(
                MODEL_ID, num_labels=2).to(DEVICE)
    model.load_state_dict(saved["best_state"])
    model.eval()

    te_dl    = DataLoader(EmailDataset(X_te, y_te, tok), batch_size=32)
    te_preds = []
    t0       = time.time()

    with torch.no_grad():
        for be, _ in te_dl:
            te_preds.extend(
                model(**{k: v.to(DEVICE) for k, v in be.items()})
                .logits.argmax(-1).cpu().numpy())

    ms = (time.time() - t0) / len(y_te) * 1000
    ALL_RESULTS.append(evaluate(y_te, te_preds, MODEL_ID, ms))

    print(f"\n{MODEL_ID}")
    print(classification_report(y_te, te_preds, target_names=["Legitimate", "Phishing"]))
    del model; torch.cuda.empty_cache()

print("\n" + "="*60)
print("NOTEBOOK 1 — ALL MODELS COMPARISON")
print("="*60)
print(pd.DataFrame(ALL_RESULTS).to_string(index=False))
print("\n→ No explanations produced — classification only.")
print("  Copy this table into the comparison cell of Notebook 2.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



bert-base-uncased
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99       324
    Phishing       0.97      0.97      0.97       134

    accuracy                           0.98       458
   macro avg       0.98      0.98      0.98       458
weighted avg       0.98      0.98      0.98       458



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



roberta-base
              precision    recall  f1-score   support

  Legitimate       0.98      1.00      0.99       324
    Phishing       0.99      0.95      0.97       134

    accuracy                           0.98       458
   macro avg       0.99      0.97      0.98       458
weighted avg       0.98      0.98      0.98       458



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



distilbert-base-uncased
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99       324
    Phishing       0.97      0.98      0.97       134

    accuracy                           0.98       458
   macro avg       0.98      0.98      0.98       458
weighted avg       0.98      0.98      0.98       458


NOTEBOOK 1 — ALL MODELS COMPARISON
                  Model Accuracy Precision Recall     F1 ms/sample
      bert-base-uncased   0.9825    0.9701 0.9701 0.9701     15.15
        bert-base-cased   0.9847    0.9635 0.9851 0.9742     15.52
           roberta-base   0.9825    0.9922 0.9478 0.9695     15.58
distilbert-base-uncased   0.9847    0.9704 0.9776 0.9740      8.26

→ No explanations produced — classification only.
  Copy this table into the comparison cell of Notebook 2.
